# 03 — Exploratory Data Analysis

Borrower-level risk patterns in the cleaned data, ahead of feature engineering. Uses `credit_risk.data` for
loading/cleaning; everything below is presentation, not reusable logic — that's why it stays inline rather
than moving into `src/`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("ggplot")

from credit_risk.config import settings
from credit_risk.data import clean_and_label, load_raw_data

In [ ]:
df = clean_and_label(load_raw_data(settings.raw_data_path))
print(df.shape)
print(f"Default rate: {df['target_default'].mean() * 100:.2f}%")

## Default rate by credit grade

In [ ]:
grade_default = df.groupby("grade")["target_default"].mean().sort_values() * 100

plt.figure(figsize=(10, 5))
grade_default.plot(kind="bar")
plt.title("Default Rate by Loan Grade")
plt.ylabel("Default Rate (%)")
plt.show()

## Default rate by sub-grade

In [ ]:
subgrade_default = df.groupby("sub_grade")["target_default"].mean().sort_values() * 100

plt.figure(figsize=(16, 6))
subgrade_default.plot(kind="bar")
plt.title("Default Rate by Sub Grade")
plt.ylabel("Default Rate (%)")
plt.show()

## Annual income

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df["annual_inc"], bins=50)
plt.xlim(0, 300000)
plt.title("Annual Income Distribution")
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(x="target_default", y="annual_inc", data=df)
plt.ylim(0, 300000)
plt.title("Income vs Default")
plt.show()

## Debt-to-income

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(x="target_default", y="dti", data=df)
plt.ylim(0, 50)
plt.title("Debt-to-Income Ratio vs Default")
plt.show()

## Home ownership

In [ ]:
home_default = df.groupby("home_ownership")["target_default"].mean().sort_values() * 100

plt.figure(figsize=(8, 5))
home_default.plot(kind="bar")
plt.title("Default Rate by Home Ownership")
plt.ylabel("Default Rate (%)")
plt.show()

## Loan purpose

In [ ]:
purpose_default = df.groupby("purpose")["target_default"].mean().sort_values() * 100

plt.figure(figsize=(14, 6))
purpose_default.plot(kind="bar")
plt.title("Default Rate by Loan Purpose")
plt.ylabel("Default Rate (%)")
plt.show()

## Revolving utilization

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(x="target_default", y="revol_util", data=df)
plt.ylim(0, 150)
plt.title("Revolving Utilization vs Default")
plt.show()

## Delinquency

In [ ]:
df.groupby("delinq_2yrs")["target_default"].mean().head(20) * 100

## Correlation structure

In [ ]:
numeric_df = df.select_dtypes(include=np.number)
corr_matrix = numeric_df.corr()

plt.figure(figsize=(14, 10))
sns.heatmap(corr_matrix, cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
target_corr = corr_matrix["target_default"].sort_values(ascending=False)
print(target_corr.head(15))
print(target_corr.tail(15))

**Findings:** lower credit grades default more; higher DTI correlates with default; certain loan purposes
carry elevated risk; revolving utilization tracks default behavior. `int_rate` is the strongest single
linear correlate with the target (~0.26) — unsurprising, since LendingClub sets `int_rate` largely *from*
its own grade/sub-grade risk assessment, so this is closer to a restatement of the grading system than an
independent signal.